In [3]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="./chroma_db")

In [5]:
import pandas as pd
import ast
import chromadb
from chromadb.utils import embedding_functions

# ---------- อ่านไฟล์ CSV ----------
df = pd.read_csv("../../../data/raw/exercisedb_all_raw_flat.csv", encoding="utf-8-sig")
print(f"โหลดข้อมูล {len(df)} รายการ")

# ---------- ฟังก์ชันแปลง string ที่เป็น list ให้เป็น list จริง ----------
def parse_list_field(value):
    """
    แปลง string '['back']' หรือ "['shoulders', 'chest']"
    ให้เป็น list จริง ๆ
    """
    if pd.isna(value) or value == "":
        return []
    try:
        # ใช้ ast.literal_eval เพื่อแปลง string -> list
        return ast.literal_eval(value)
    except:
        # ถ้าแปลงไม่ได้ ให้คืนค่าเป็น list ว่าง
        return []

# ---------- สร้าง document และ metadata สำหรับแต่ละแถว ----------
documents = []
metadatas = []
ids = []

for idx, row in df.iterrows():
    # 1. สร้าง exercise_id
    exercise_id = row["exerciseId"]
    
    # 2. แปลงฟิลด์ที่เป็น list string
    body_parts = parse_list_field(row["bodyParts"])
    equipments = parse_list_field(row["equipments"])
    target_muscles = parse_list_field(row["targetMuscles"])
    secondary_muscles = parse_list_field(row["secondaryMuscles"])
    instructions = parse_list_field(row["instructions"])
    
    # 3. สร้าง document text
    #    ใช้ชื่อท่า + รายละเอียด
    doc_text = row["name"]
    if instructions:
        doc_text += "\n\n" + "\n".join(instructions)
    
    # 4. เตรียม metadata
    #    ChromaDB รับได้เฉพาะ str, int, float, bool
    #    ดังนั้นต้องแปลง list เป็น string คั่นด้วย comma
    metadata = {
        "exercise_id": exercise_id,
        "name": row["name"],
        "body_parts": ", ".join(body_parts) if body_parts else "",
        "equipment": ", ".join(equipments) if equipments else "",
        "target_muscles": ", ".join(target_muscles) if target_muscles else "",
        "secondary_muscles": ", ".join(secondary_muscles) if secondary_muscles else "",
        "source": "exercisedb"
    }
    
    # 5. เก็บลง list
    documents.append(doc_text)
    metadatas.append(metadata)
    ids.append(exercise_id)

print(f"เตรียมข้อมูล {len(documents)} รายการ")

โหลดข้อมูล 1500 รายการ
เตรียมข้อมูล 1500 รายการ


In [6]:
# ---------- สร้าง ChromaDB client ----------
client = chromadb.PersistentClient(path="./chroma_db")

# ---------- กำหนด embedding function ----------
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ---------- สร้าง collection ----------
collection = client.get_or_create_collection(
    name="exercise_instructions",
    embedding_function=embedding_fn,
    metadata={"description": "ExerciseDB 1,500+ instructions with metadata"}
)

# ---------- แบ่ง batch และเพิ่มข้อมูล ----------
BATCH_SIZE = 100

for i in range(0, len(documents), BATCH_SIZE):
    batch_docs = documents[i:i+BATCH_SIZE]
    batch_metas = metadatas[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]
    
    collection.add(
        documents=batch_docs,
        metadatas=batch_metas,
        ids=batch_ids
    )
    
    print(f"✅ เพิ่มแล้ว {min(i+BATCH_SIZE, len(documents))}/{len(documents)}")

print(f"✅ นำเข้าเสร็จ! มีทั้งหมด {collection.count()} รายการใน collection")

c:\Users\jirap\miniconda3\envs\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jirap\miniconda3\envs\ML\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jirap\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to 

✅ เพิ่มแล้ว 100/1500
✅ เพิ่มแล้ว 200/1500
✅ เพิ่มแล้ว 300/1500
✅ เพิ่มแล้ว 400/1500
✅ เพิ่มแล้ว 500/1500
✅ เพิ่มแล้ว 600/1500
✅ เพิ่มแล้ว 700/1500
✅ เพิ่มแล้ว 800/1500
✅ เพิ่มแล้ว 900/1500
✅ เพิ่มแล้ว 1000/1500
✅ เพิ่มแล้ว 1100/1500
✅ เพิ่มแล้ว 1200/1500
✅ เพิ่มแล้ว 1300/1500
✅ เพิ่มแล้ว 1400/1500
✅ เพิ่มแล้ว 1500/1500
✅ นำเข้าเสร็จ! มีทั้งหมด 1500 รายการใน collection


In [10]:
import re
import chromadb
from chromadb.utils import embedding_functions

# 1. เชื่อมต่อกับ ChromaDB (ใช้ path เดิม)
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("exercise_instructions")  # ใช้ collection เดียวกัน

# 2. อ่านไฟล์ RAG_CORPUS.md
file_path = "../RAG_CORPUS.md"
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# 3. ใช้ Regular Expression แยกข้อมูลแต่ละ Rule
#    รูปแบบ: ### rule_id \n Category: ... \n Applies to: ... \n Rule: ... \n Use in system: ...
pattern = r"### (\w+)\nCategory: (.+?)\nApplies to: (.+?)\nRule: (.+?)\nUse in system: (.+?)(?=\n---\n|\n###|\Z)"

matches = re.findall(pattern, content, re.DOTALL)

print(f"✅ พบกฎทั้งหมด {len(matches)} ข้อ")

# 4. เตรียมข้อมูลสำหรับ ChromaDB
rule_documents = []
rule_metadatas = []
rule_ids = []

for rule_id, category, applies_to, rule_text, use_in_system in matches:
    # ทำความสะอาดข้อความ (ตัดช่องว่าง)
    rule_id = rule_id.strip()
    category = category.strip()
    applies_to = applies_to.strip()
    rule_text = rule_text.strip()
    
    # สร้าง Document (ข้อความที่ใช้ค้นหา) 
    # ใช้ทั้งเนื้อหากฎ + เงื่อนไข เพื่อให้ค้นหาเจอง่ายขึ้น
    doc_content = f"{rule_text} (Condition: {applies_to})"
    
    # เตรียม Metadata
    # ต้องแปลง list ให้เป็น string เท่านั้น เพราะ ChromaDB ไม่รับ list โดยตรง
    metadata = {
        "source": "rule_corpus",
        "category": category,
        "applies_to": applies_to,
        "rule_text": rule_text,  # เก็บเนื้อหากฎไว้ใน metadata เผื่อใช้งาน
        "use_in_system": use_in_system.strip()
    }
    
    rule_documents.append(doc_content)
    rule_metadatas.append(metadata)
    rule_ids.append(rule_id)

# 5. เพิ่มข้อมูลลง ChromaDB (ใช้ Batch เพื่อความปลอดภัย)
BATCH_SIZE = 50
for i in range(0, len(rule_documents), BATCH_SIZE):
    collection.add(
        documents=rule_documents[i:i+BATCH_SIZE],
        metadatas=rule_metadatas[i:i+BATCH_SIZE],
        ids=rule_ids[i:i+BATCH_SIZE]
    )
    print(f"✅ เพิ่มกฎแล้ว {min(i+BATCH_SIZE, len(rule_documents))}/{len(rule_documents)}")

print(f"\n🎉 นำเข้า Rule Corpus สำเร็จ! ตอนนี้ Collection มีทั้งหมด {collection.count()} รายการ")

✅ พบกฎทั้งหมด 80 ข้อ
✅ เพิ่มกฎแล้ว 50/80
✅ เพิ่มกฎแล้ว 80/80

🎉 นำเข้า Rule Corpus สำเร็จ! ตอนนี้ Collection มีทั้งหมด 1580 รายการ


In [11]:
# เช็คจำนวนกฎที่เพิ่มเข้าไป
rule_check = collection.get(
    where={"source": "rule_corpus"},
    include=["metadatas"]
)
print(f"จำนวนกฎในระบบ: {len(rule_check['ids'])} ข้อ")

# แสดงตัวอย่างกฎ 3 ข้อแรก
for i, meta in enumerate(rule_check['metadatas'][:3], 1):
    print(f"{i}. {meta['category']} - {meta['rule_text'][:50]}...")

จำนวนกฎในระบบ: 80 ข้อ
1. Recovery - If sleep duration is below 6 hours, reduce trainin...
2. Recovery - High stress should reduce workout intensity and pr...
3. Recovery - The same major muscle group should not be trained ...


In [12]:
test_queries = [
    "ถ้านอนน้อยกว่า 6 ชั่วโมงควรออกกำลังกายยังไง",
    "ท่าบริหารหน้าอกด้วยดัมเบล",
    "ถ้าความดันสูงควรออกกำลังกายยังไง",
    "ท่าที่ใช้แค่ body weight",
    "มือใหม่หัดเล่นกล้ามเนื้อหลังควรทำท่าไหน"
]

for q in test_queries:
    print(f"\n🔍 คำถาม: {q}")
    print("-" * 60)
    
    results = collection.query(
        query_texts=[q],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )
    
    for i, (doc_id, doc, meta, dist) in enumerate(zip(
        results['ids'][0],
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        source = meta.get('source', 'unknown')
        name_or_rule = meta.get('name', meta.get('rule_text', ''))[:50]
        
        print(f"  Rank {i} (Score: {1-dist:.4f})")
        print(f"    Source: {source}")
        print(f"    Content: {name_or_rule}...")
        print(f"    Category/Equipment: {meta.get('category', meta.get('equipment', 'N/A'))}")


🔍 คำถาม: ถ้านอนน้อยกว่า 6 ชั่วโมงควรออกกำลังกายยังไง
------------------------------------------------------------
  Rank 1 (Score: 0.2099)
    Source: exercisedb
    Content: chin-up diagonal...
    Category/Equipment: body weight
  Rank 2 (Score: 0.2003)
    Source: exercisedb
    Content: mixed grip chin-up...
    Category/Equipment: body weight
  Rank 3 (Score: 0.1964)
    Source: exercisedb
    Content: chin-ups (narrow parallel grip)...
    Category/Equipment: body weight
  Rank 4 (Score: 0.1956)
    Source: exercisedb
    Content: chin-up...
    Category/Equipment: body weight
  Rank 5 (Score: 0.1912)
    Source: exercisedb
    Content: one arm chin-up...
    Category/Equipment: body weight

🔍 คำถาม: ท่าบริหารหน้าอกด้วยดัมเบล
------------------------------------------------------------
  Rank 1 (Score: 0.1598)
    Source: exercisedb
    Content: rounded twisted leg raise (female)...
    Category/Equipment: body weight
  Rank 2 (Score: 0.1590)
    Source: exercisedb
    Content: 